# LIAR Dataset — Exploratory Data Analysis

LIAR is a 6-way truthfulness classification dataset sourced from PolitiFact.  
12,836 short political statements labelled by expert journalists.

**Labels:** `pants-fire` → `false` → `barely-true` → `half-true` → `mostly-true` → `true`  
**Binary mapping used in benchmark:** `{pants-fire, false, barely-true}` = fake (1), `{mostly-true, true}` = real (0)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = Path('../datasets/LIARDataset')

COLUMNS = [
    'statement_id', 'label', 'statement', 'subjects',
    'speaker', 'speaker_job_title', 'state_info', 'party_affiliation',
    'barely_true_count', 'false_count', 'half_true_count',
    'mostly_true_count', 'pants_on_fire_count', 'context',
]

train = pd.read_csv(DATA_DIR / 'train.tsv', sep='\t', header=None, names=COLUMNS)
valid = pd.read_csv(DATA_DIR / 'valid.tsv', sep='\t', header=None, names=COLUMNS)
test  = pd.read_csv(DATA_DIR / 'test.tsv',  sep='\t', header=None, names=COLUMNS)

for split, df in [('train', train), ('valid', valid), ('test', test)]:
    print(f'{split:6s}: {len(df):,} rows')

df = pd.concat([train, valid, test], keys=['train', 'valid', 'test']).reset_index(level=0).rename(columns={'level_0': 'split'})

## 1. Label Distribution

In [ ]:
LABEL_ORDER = ['pants-fire', 'false', 'barely-true', 'half-true', 'mostly-true', 'true']
LABEL_COLORS = ['#d73027', '#f46d43', '#fdae61', '#abd9e9', '#74add1', '#313695']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 6-way counts
counts = df['label'].value_counts().reindex(LABEL_ORDER)
axes[0].bar(LABEL_ORDER, counts.values, color=LABEL_COLORS)
axes[0].set_title('6-way Label Distribution (all splits)')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=9)

# Binary
fake_labels = {'pants-fire', 'false', 'barely-true'}
df['binary'] = df['label'].apply(lambda x: 'fake' if x in fake_labels else ('real' if x in {'mostly-true', 'true'} else 'borderline (half-true)'))
binary_counts = df['binary'].value_counts()
axes[1].pie(binary_counts, labels=binary_counts.index, autopct='%1.1f%%',
            colors=['#d73027', '#313695', '#fdae61'], startangle=90)
axes[1].set_title('Binary split (fake / real / borderline)')

plt.tight_layout()
plt.show()
print('\nCounts:')
print(counts.to_string())

## 2. Label Distribution per Split

In [ ]:
split_label = df.groupby(['split', 'label']).size().unstack().reindex(columns=LABEL_ORDER)
split_label.plot(kind='bar', figsize=(10, 5), color=LABEL_COLORS)
plt.title('Label counts per split')
plt.xlabel('Split')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Label', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

## 3. Statement Length Distribution

In [ ]:
df['char_len'] = df['statement'].str.len()
df['word_len'] = df['statement'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ['char_len', 'word_len'], ['Characters', 'Words']):
    for lbl, color in zip(LABEL_ORDER, LABEL_COLORS):
        subset = df[df['label'] == lbl][col]
        ax.hist(subset, bins=30, alpha=0.5, label=lbl, color=color)
    ax.set_title(f'Statement length ({label})')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

print(df.groupby('label')[['char_len', 'word_len']].median().reindex(LABEL_ORDER).round(1))

## 4. Top Speakers

In [ ]:
top_speakers = df['speaker'].value_counts().head(15)
top_speakers.plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Top 15 speakers by statement count')
plt.xlabel('Number of statements')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Party Affiliation vs Label

In [ ]:
top_parties = df['party_affiliation'].value_counts().head(6).index
party_df = df[df['party_affiliation'].isin(top_parties)]
party_label = party_df.groupby(['party_affiliation', 'label']).size().unstack().fillna(0).reindex(columns=LABEL_ORDER)
party_label_pct = party_label.div(party_label.sum(axis=1), axis=0) * 100

party_label_pct.plot(kind='bar', stacked=True, figsize=(10, 5), color=LABEL_COLORS)
plt.title('Label distribution by party affiliation (% stacked)')
plt.xlabel('Party')
plt.ylabel('% of statements')
plt.xticks(rotation=15)
plt.legend(title='Label', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

## 6. Speaker Credibility (derived from credit history)

In [ ]:
count_cols = ['barely_true_count', 'false_count', 'half_true_count', 'mostly_true_count', 'pants_on_fire_count']
df['total_history'] = df[count_cols].sum(axis=1)
df['speaker_cred'] = (df['mostly_true_count'] + df['half_true_count'] * 0.5) / df['total_history'].replace(0, float('nan'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['speaker_cred'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Speaker credibility score distribution')
axes[0].set_xlabel('Credibility (0=dishonest, 1=honest)')
axes[0].set_ylabel('Count')

cred_by_label = df.groupby('label')['speaker_cred'].median().reindex(LABEL_ORDER)
axes[1].bar(LABEL_ORDER, cred_by_label, color=LABEL_COLORS)
axes[1].set_title('Median speaker credibility by label')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Median credibility')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()
print('Speaker credibility → strongly correlated with label — good signal!')

## 7. Top Subjects / Topics

In [ ]:
topics = df['subjects'].dropna().str.split(',').explode().str.strip()
top_topics = topics.value_counts().head(20)

top_topics.plot(kind='barh', figsize=(8, 6), color='teal')
plt.title('Top 20 topics / subjects')
plt.xlabel('Statement count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Statement context (venue)

In [ ]:
top_contexts = df['context'].value_counts().head(15)
top_contexts.plot(kind='barh', figsize=(8, 5), color='coral')
plt.title('Top 15 statement contexts (venue)')
plt.xlabel('Count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Benchmark readiness summary

| Property | Value |
|---|---|
| Total records | 12,836 |
| Test split | 1,267 |
| Labels | 6-way (collapsed to binary for eval) |
| Has images | No |
| Has speaker history | Yes — used as source credibility prior |
| Hardest classes | `half-true`, `mostly-true` — boundary cases |
| Pipeline input | `statement` column → `claim_text` |
| Source URL | Constructed from speaker name (PolitiFact format) |

In [ ]:
print('Test split label counts:')
print(test['label'].value_counts().reindex(LABEL_ORDER))
print(f'\nTest binary: {(test["label"].isin(["pants-fire","false","barely-true"])).sum()} fake, '
      f'{(test["label"].isin(["mostly-true","true"])).sum()} real, '
      f'{(test["label"]=="half-true").sum()} borderline')